# 06 - So sánh ML Models: Tabular vs Graph vs Combined

## Phát hiện Giao dịch Gian lận dựa trên Mạng Neural Đồ thị

**7 Models**: Decision Tree, Random Forest, LightGBM, KNN, Naïve Bayes, Bagging, XGBoost

**3 Scenarios**:
1. **Tabular Only** – C, D, V columns + amount + time
2. **Graph Only** – Entity degree, amount stats, time gaps, diversity
3. **Combined** – Tabular + Graph features

**Tối ưu CPU**: Subsample 30%, KNN giới hạn 80K rows, parallel n_jobs=-1

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import polars as pl
import matplotlib
matplotlib.use('TkAgg')  # hoặc 'Qt5Agg' trên Windows
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import set_seed, SEED
set_seed(42)
print('Ready.')

Ready.


## 1. Load Data (Subsample 30% cho tốc độ)

In [2]:
from src.config import PATH_TRAIN_TRANSACTION, PATH_TRAIN_IDENTITY

# === Expanded columns (giống Colab v2) ===
C_COLS = [f'C{i}' for i in range(1, 15)]
D_COLS = [f'D{i}' for i in range(1, 16)]
V_COLS = [f'V{i}' for i in range(12, 40)]

USECOLS_TRAN = [
    'TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt',
    'ProductCD',
    'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'addr2',
    'P_emaildomain', 'R_emaildomain',
] + C_COLS + D_COLS + V_COLS

USECOLS_ID = ['TransactionID', 'DeviceType', 'DeviceInfo', 'id_30', 'id_31']

# Load
lt = pl.scan_csv(str(PATH_TRAIN_TRANSACTION), infer_schema_length=5000)
avail = lt.collect_schema().names()
use_tran = [c for c in USECOLS_TRAN if c in avail]
lt = lt.select(use_tran)
li = pl.scan_csv(str(PATH_TRAIN_IDENTITY), infer_schema_length=5000).select(USECOLS_ID)

df = lt.join(li, on='TransactionID', how='left').collect(streaming=False)

# Subsample 30% for CPU speed
SUBSAMPLE = 0.30
df = df.sample(fraction=SUBSAMPLE, seed=SEED)
df = df.with_row_index(name='txn_index')

n = df.height
fr = int(df['isFraud'].sum())
print(f'Loaded {n:,} transactions ({SUBSAMPLE*100:.0f}% sample)')
print(f'Fraud: {fr:,} ({fr/n*100:.2f}%)')
print(f'Columns: {len(df.columns)}')

C:\Users\Tuan\AppData\Local\Temp\ipykernel_2532\2367434177.py:25: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df = lt.join(li, on='TransactionID', how='left').collect(streaming=False)


Loaded 177,162 transactions (30% sample)
Fraud: 6,218 (3.51%)
Columns: 77


## 2. Time-based Train/Test Split

In [3]:
TRAIN_RATIO = 0.80

time_col = df['TransactionDT'].cast(pl.Float64)
threshold = float(time_col.quantile(TRAIN_RATIO))
times = time_col.fill_null(0.0).to_numpy()

train_mask = times <= threshold
test_mask = times > threshold

y_all = df.select(pl.col('isFraud').fill_null(0).cast(pl.Int64))['isFraud'].to_numpy()
y_train = y_all[train_mask]
y_test = y_all[test_mask]

print(f'Train: {train_mask.sum():,} ({y_train.sum():,} fraud, {y_train.mean()*100:.2f}%)')
print(f'Test:  {test_mask.sum():,} ({y_test.sum():,} fraud, {y_test.mean()*100:.2f}%)')

Train: 141,730 (4,989 fraud, 3.52%)
Test:  35,432 (1,229 fraud, 3.47%)


## 3. Build Tabular Features

In [4]:
def build_tabular_features(df, use_tran):
    """Build tabular features matching Colab v2 structure."""
    features = []
    
    # Amount
    amt = df['TransactionAmt'].cast(pl.Float64).fill_null(0.0).to_numpy()
    mu, sd = float(np.mean(amt)), max(float(np.std(amt)), 1.0)
    features.append((amt - mu) / (sd + 1e-6))
    features.append(np.log1p(amt))
    
    # Time
    sec = df['TransactionDT'].cast(pl.Float64).fill_null(0.0).to_numpy()
    hour = (sec % 86400.0) / 3600.0
    dow = ((sec // 86400.0) % 7.0)
    features.extend([
        np.sin(2*np.pi*hour/24), np.cos(2*np.pi*hour/24),
        np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7),
    ])
    
    # C, D, V columns
    for col in C_COLS + D_COLS + V_COLS:
        if col in use_tran:
            arr = df[col].cast(pl.Float64).fill_null(0.0).to_numpy()
            if col.startswith('C'):
                arr = np.log1p(arr)
            features.append(arr)
    
    X = np.stack(features, axis=1).astype(np.float32)
    X = np.nan_to_num(X, nan=0.0)
    print(f'Tabular features: {X.shape}')
    return X

X_tab = build_tabular_features(df, use_tran)
X_tab_train = X_tab[train_mask]
X_tab_test = X_tab[test_mask]

Tabular features: (177162, 63)


## 4. Build Graph Features

In [5]:
from src.graph_features import GraphFeatureExtractor

gfe = GraphFeatureExtractor()
X_graph = gfe.fit_transform(df, train_mask)

X_graph_train = X_graph[train_mask]
X_graph_test = X_graph[test_mask]

print(f'\nGraph features: {X_graph.shape[1]} dims')
print(f'Feature names: {gfe.feature_names}')

Extracting graph features...
  Graph features: 21 dimensions
  Feature names: ['degree_card1', 'degree_card2', 'degree_card3', 'degree_addr1', 'degree_addr2', 'degree_P_emaildomain', 'degree_R_emaildomain', 'degree_DeviceType', 'degree_DeviceInfo', 'amt_mean_card1', 'amt_std_card1', 'amt_dev_card1', 'amt_mean_addr1', 'amt_std_addr1', 'amt_dev_addr1', 'amt_mean_P_emaildomain', 'amt_std_P_emaildomain', 'amt_dev_P_emaildomain', 'time_gap_card1', 'txn_count_card1', 'entity_diversity']
[fit_transform] completed in 1.03s

Graph features: 21 dims
Feature names: ['degree_card1', 'degree_card2', 'degree_card3', 'degree_addr1', 'degree_addr2', 'degree_P_emaildomain', 'degree_R_emaildomain', 'degree_DeviceType', 'degree_DeviceInfo', 'amt_mean_card1', 'amt_std_card1', 'amt_dev_card1', 'amt_mean_addr1', 'amt_std_addr1', 'amt_dev_addr1', 'amt_mean_P_emaildomain', 'amt_std_P_emaildomain', 'amt_dev_P_emaildomain', 'time_gap_card1', 'txn_count_card1', 'entity_diversity']


## 5. Train 7 Models × 3 Scenarios

In [6]:
from src.ml_trainer import MLTrainer

trainer = MLTrainer(subsample_knn=80_000)

all_results = trainer.run_all_scenarios(
    X_tab_train, X_tab_test,
    X_graph_train, X_graph_test,
    y_train, y_test,
)


SCENARIO: Tabular Only
  Train: (141730, 63), Test: (35432, 63)
  Fraud train: 4,989/141,730 (3.52%)
  Fraud test:  1,229/35,432 (3.47%)

  Training Decision Tree... F1=0.2157 AUC=0.7957 (3.0s)

  Training Random Forest... F1=0.3033 AUC=0.8668 (7.7s)

  Training KNN... F1=0.1606 AUC=0.6770 (42.9s)

  Training Naive Bayes... F1=0.1901 AUC=0.7754 (0.2s)

  Training Bagging... F1=0.4515 AUC=0.8278 (24.1s)

  Training LightGBM... F1=0.3398 AUC=0.8615 (3.3s)

  Training XGBoost... F1=0.3781 AUC=0.8496 (4.1s)

SCENARIO: Graph Only
  Train: (141730, 21), Test: (35432, 21)
  Fraud train: 4,989/141,730 (3.52%)
  Fraud test:  1,229/35,432 (3.47%)

  Training Decision Tree... F1=0.1627 AUC=0.7329 (1.9s)

  Training Random Forest... F1=0.2879 AUC=0.8045 (6.7s)

  Training KNN... F1=0.0979 AUC=0.6791 (16.6s)

  Training Naive Bayes... F1=0.1837 AUC=0.7078 (0.1s)

  Training Bagging... F1=0.1965 AUC=0.7715 (12.0s)

  Training LightGBM... F1=0.2578 AUC=0.7911 (2.3s)

  Training XGBoost... F1=0.3000 

## 6. Bảng so sánh

In [7]:
trainer.print_comparison()


ML MODEL COMPARISON ACROSS SCENARIOS

───────────────────────────────────────────────────────────────────────────
  Tabular Only
───────────────────────────────────────────────────────────────────────────
  Model               Precision     Recall         F1    AUC-ROC     AUC-PR
  ────────────────────────────────────────────────────────────────────
  Decision Tree          0.1293     0.6509     0.2157     0.7957     0.3849
  Random Forest          0.1973     0.6550     0.3033     0.8668     0.4347
  KNN                    0.5665     0.0936     0.1606     0.6770     0.1592
  Naive Bayes            0.1158     0.5297     0.1901     0.7754     0.1182
  Bagging                0.7773     0.3181     0.4515     0.8278     0.4278
  LightGBM               0.2350     0.6135     0.3398     0.8615     0.4432
  XGBoost                0.3015     0.5069     0.3781     0.8496     0.4339

───────────────────────────────────────────────────────────────────────────
  Graph Only
─────────────────────────

## 7. Visualizations

In [8]:
from pathlib import Path
out_dir = Path('../output/metrics')
out_dir.mkdir(parents=True, exist_ok=True)

scenarios = list(all_results.keys())
models = list(all_results[scenarios[0]].keys())

# === F1 Comparison Bar Chart ===
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(models))
width = 0.25
colors = ['#3498db', '#e74c3c', '#2ecc71']

for i, (scenario, color) in enumerate(zip(scenarios, colors)):
    f1s = [all_results[scenario][m]['f1'] for m in models]
    bars = ax.bar(x + i*width, f1s, width, label=scenario, color=color, edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, f1s):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', fontsize=7, fontweight='bold')

ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('F1 Score: Tabular vs Graph vs Combined', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(models, rotation=30, ha='right', fontsize=10)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(str(out_dir / 'ml_f1_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_dir / "ml_f1_comparison.png"}')

Saved: ..\output\metrics\ml_f1_comparison.png


In [9]:
# === AUC-ROC Comparison ===
fig, ax = plt.subplots(figsize=(14, 6))
for i, (scenario, color) in enumerate(zip(scenarios, colors)):
    aucs = [all_results[scenario][m]['auc_roc'] for m in models]
    bars = ax.bar(x + i*width, aucs, width, label=scenario, color=color, edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, aucs):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', fontsize=7, fontweight='bold')

ax.set_ylabel('AUC-ROC', fontsize=12)
ax.set_title('AUC-ROC: Tabular vs Graph vs Combined', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(models, rotation=30, ha='right', fontsize=10)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(str(out_dir / 'ml_auc_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [10]:
# === ROC Curves for best scenario (Combined) ===
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(10, 7))
cmap = plt.cm.Set1(np.linspace(0, 1, len(models)))

for (name, r), color in zip(all_results['Combined'].items(), cmap):
    if r['auc_roc'] > 0:
        fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
        ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={r['auc_roc']:.4f})")

ax.plot([0,1], [0,1], 'k--', lw=1)
ax.set_xlabel('FPR', fontsize=12)
ax.set_ylabel('TPR', fontsize=12)
ax.set_title('ROC Curves - Combined Features', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(str(out_dir / 'ml_roc_combined.png'), dpi=150, bbox_inches='tight')
plt.show()

In [11]:
# === Confusion Matrices for Combined (top 3 models) ===
from sklearn.metrics import confusion_matrix

combined = all_results['Combined']
top3 = sorted(combined.keys(), key=lambda k: combined[k]['f1'], reverse=True)[:3]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name in zip(axes, top3):
    cm = confusion_matrix(y_test, combined[name]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Non-Fraud', 'Fraud'],
                yticklabels=['Non-Fraud', 'Fraud'], ax=ax)
    ax.set_title(f"{name}\nF1={combined[name]['f1']:.4f}", fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices - Combined Features (Top 3)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(str(out_dir / 'ml_confusion_combined.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Kết luận

Bảng tổng kết F1-Score:

| Model | Tabular Only | Graph Only | Combined |
|-------|-------------|------------|----------|
| Decision Tree | - | - | - |
| Random Forest | - | - | - |
| LightGBM | - | - | - |
| KNN | - | - | - |
| Naïve Bayes | - | - | - |
| Bagging | - | - | - |
| XGBoost | - | - | - |

*Điền kết quả sau khi chạy.*

**Nhận xét dự kiến:**
- Combined features cho kết quả tốt nhất
- Graph features bổ sung thông tin topology mà tabular không có
- LightGBM/XGBoost thường là top performers